# AI4CARE Judge Benchmark Analysis

Automated notebook analyzing the performance and agreement of the selected MoE judges.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid")

LONG_CSV = "../results/judge_benchmark/judge_scores_long.csv"
AGGR_CSV = "../results/judge_benchmark/panel_aggregated.csv"

# Load datasets
long_df = pd.read_csv(LONG_CSV) if os.path.exists(LONG_CSV) else pd.DataFrame()
aggr_df = pd.read_csv(AGGR_CSV) if os.path.exists(AGGR_CSV) else pd.DataFrame()

if not long_df.empty:
    print(f"Loaded {len(long_df)} rows from {LONG_CSV}")
    print("Sanity Check: missing values?")
    print(long_df.isna().sum())
    
    g = sns.FacetGrid(long_df, col="judge_id", row="metric", margin_titles=True, sharex=False, sharey=False)
    g.map(sns.histplot, "score", bins=10)
    g.fig.suptitle("Score Distributions per Judge & Metric (Sanity)", y=1.02)
    plt.show()

In [ ]:
# Cell 2: Judge Agreement Heatmap
if not long_df.empty:
    metrics = long_df["metric"].unique()
    for m in metrics:
        subset = long_df[long_df["metric"] == m].copy()
        # Pivot so that columns are judges, rows are traces
        pivot = subset.pivot_table(index="run_id", columns="judge_id", values="score")
        
        corr = pivot.corr(method="spearman")
        
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
        plt.title(f"Pairwise Spearman Correlation - {m}")
        plt.show()

In [ ]:
# Cell 3: Ranking Comparison
EVAL_RESULTS = "../results/evaluation_results.csv"

if not long_df.empty and os.path.exists(EVAL_RESULTS):
    eval_df = pd.read_csv(EVAL_RESULTS)
    merged_long = pd.merge(long_df, eval_df[["run_id"]], on="run_id", how="inner")
    
    # Bar chart grouping by candidate model
    for m in ["overall_elicit_efficiency"]:
        if m in merged_long["metric"].values:
            plt.figure(figsize=(10,6))
            sns.barplot(data=merged_long[merged_long["metric"] == m], x="candidate_model_id", y="score", hue="judge_id", capsize=.1)
            plt.title(f"Mean {m} by Candidate Model Across Judges")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

In [ ]:
# Cell 4: Does the Mixture Change Conclusions?
if not long_df.empty and not aggr_df.empty:
    # For simplicity, treat the max judge score as 'single-best'
    best_judges = long_df.groupby(["run_id", "metric"])["score"].max().reset_index(name="best_judge_score")
    merged = pd.merge(best_judges, aggr_df, on=["run_id", "metric"])
    
    if not merged.empty:
        plt.figure(figsize=(8,8))
        sns.scatterplot(data=merged, x="best_judge_score", y="panel_median", hue="metric")
        # Draw identity line
        plt.plot([0, 10], [0, 10], "k--", alpha=0.5)
        plt.title("Single-Best-Judge vs Panel Median")
        plt.xlabel("Best Individual Judge Score")
        plt.ylabel("Panel Median Score")
        plt.tight_layout()
        plt.show()
        
        diff = np.abs(merged["best_judge_score"] - merged["panel_median"])
        off_diag = (diff > 0.5).sum()
        total = len(diff)
        print(f"Cases off-diagonal by > 0.5: {off_diag}/{total} ({off_diag/total:.1%})")